#### **Setup, Imports, and Data Loading**

In [5]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df_master = pd.read_csv('../data/processed/master_analytics_dataset.csv')
print(f"Loaded master dataset shape: {df_master.shape}")

Loaded master dataset shape: (4525, 28)


#### **Aggregate agent-level performance features for clustering**

In [ ]:
group_col = 'agent_id' if 'agent_id' in df_master.columns else ('agent_name' if 'agent_name' in df_master.columns else 'agent_tier')

agent_profile = df_master.groupby(group_col).agg(
    total_orders=('order_id', 'count'),
    mean_mortality_rate=('mortality_rate_pct', 'mean'),
    mean_vaccination_compliance=('vaccination_compliance_pct', 'mean'),
    mean_quantity_ordered=('quantity_ordered', 'mean'),
    agent_tier=('agent_tier', 'first')
).reset_index()

print(f"Aggregated agent profile shape: {agent_profile.shape}")
display(agent_profile.head())

Aggregated agent profile shape: (500, 6)


,agent_id,total_orders,mean_mortality_rate,mean_vaccination_compliance,mean_quantity_ordered,agent_tier
0,AGT-0001,10,4.793000,87.400000,290.000000,Bronze
1,AGT-0002,6,5.528333,90.166667,183.333333,Silver
2,AGT-0003,8,4.687500,88.375000,187.500000,Gold
3,AGT-0004,9,3.370000,88.222222,222.222222,Gold
4,AGT-0005,10,6.867000,89.800000,160.000000,Bronze


#### **Scale features and evaluate K-Means clustering performance across a range of k values**

In [7]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Select numerical features for agent clustering
cluster_features = [
    'total_orders', 
    'mean_mortality_rate', 
    'mean_vaccination_compliance', 
    'mean_quantity_ordered'
]
X_cluster = agent_profile[cluster_features].copy()

scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

# Evaluate K-means across a range of k values
sse = []
sil_scores = []
K_range = range(2, 8)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster_scaled)
    sse.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_cluster_scaled, kmeans.labels_))

for k, s, sil in zip(K_range, sse, sil_scores):
    print(f"k={k} | SSE: {s:.2f} | Silhouette Score: {sil:.4f}")

k=2 | SSE: 1148.34 | Silhouette Score: 0.4928
k=3 | SSE: 828.94 | Silhouette Score: 0.3550
k=4 | SSE: 608.25 | Silhouette Score: 0.3792
k=5 | SSE: 540.60 | Silhouette Score: 0.3492
k=6 | SSE: 484.93 | Silhouette Score: 0.3083
k=7 | SSE: 437.76 | Silhouette Score: 0.2700


#### **Fit K-Means with k=4 and generate agent cluster summary profiles**

In [8]:
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
agent_profile['cluster'] = kmeans.fit_predict(X_cluster_scaled)

cluster_summary = agent_profile.groupby('cluster').agg(
    agent_count=('agent_id', 'count'),
    mean_total_orders=('total_orders', 'mean'),
    mean_mortality_rate=('mean_mortality_rate', 'mean'),
    mean_vaccination_compliance=('mean_vaccination_compliance', 'mean'),
    mean_quantity_ordered=('mean_quantity_ordered', 'mean'),
    dominant_tier=('agent_tier', lambda x: x.mode()[0] if not x.mode().empty else 'Mixed')
).reset_index()

print("--- Agent Cluster Profiles (k=4) ---")
print(cluster_summary.to_string(index=False))

--- Agent Cluster Profiles (k=4) ---
 cluster  agent_count  mean_total_orders  mean_mortality_rate  mean_vaccination_compliance  mean_quantity_ordered dominant_tier
       0          156          11.051282             5.486515                    87.394567             253.671976        Bronze
       1           96           9.020833            12.198299                    69.707415             255.450111           New
       2          152           7.250000             5.914687                    87.447160             230.684785        Bronze
       3           96           8.677083             5.833717                    87.917384             363.865064        Silver


#### **Map cluster IDs to descriptive operational segments, merge with the master dataset, and save processed output files**

In [ ]:
cluster_labels = {
    1: 'High-Risk / New Onboarding',
    0: 'Standard Bronze Group A',
    2: 'Standard Bronze Group B',
    3: 'High-Volume Silver'
}
agent_profile['segment_name'] = agent_profile['cluster'].map(cluster_labels)

# Merge cluster segment back into master dataset
df_segmented = df_master.merge(
    agent_profile[['agent_id', 'cluster', 'segment_name']], 
    on='agent_id', 
    how='left'
)

output_path = '../data/processed/master_analytics_segmented.csv'
df_segmented.to_csv(output_path, index=False)
agent_profile.to_csv('../data/processed/agent_cluster_profiles.csv', index=False)
print(f"Saved segmented master dataset to {output_path}")
print(f"Saved agent cluster summary to ../data/processed/agent_cluster_profiles.csv")

Saved segmented master dataset to ../data/processed/master_analytics_segmented.csv
Saved agent cluster summary to ../data/processed/agent_cluster_profiles.csv


#### **Scale agent features and apply K-Means clustering with k=3 to compute cluster mean profiles**

In [4]:
features = ['total_volume', 'avg_mortality_rate', 'avg_vax_compliance', 'avg_dso']
X = df_agents_agg[features].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_agents_agg['cluster_id'] = kmeans.fit_predict(X_scaled)

df_agents_agg.groupby('cluster_id')[features].mean().round(2)

,total_volume,avg_mortality_rate,avg_vax_compliance,avg_dso
cluster_id,,,,
0,1918.30,5.90,87.49,12.81
1,3138.89,5.52,87.60,16.46
2,2314.58,12.20,69.71,14.74


#### **Save the segmented agent dataset to CSV**

In [5]:
processed_dir = '../data/processed'
os.makedirs(processed_dir, exist_ok=True)
df_agents_agg.to_csv(os.path.join(processed_dir, 'segmented_agents.csv'), index=False)

print(f"Agent segmentation complete and saved to {processed_dir}/segmented_agents.csv!")

Agent segmentation complete and saved to ../data/processed/segmented_agents.csv!
